<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 5

*Fine-tune CLIP.*

## Imports and initializations

In [ ]:
import gc
import torch
import requests
import numpy as np
import torchvision
import transformers
from PIL import Image
import torch.nn.functional as F
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import Counter
from torchvision import transforms
from sklearn import metrics, preprocessing
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

In [ ]:
BATCH_SIZE = 128
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
imagenette = torchvision.datasets.Imagenette('imagenette/', download=True)
imagenette_train, imagenette_test = torch.utils.data.random_split(imagenette, [0.75, 0.25])

train_loader = torch.utils.data.DataLoader(imagenette_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))
test_loader = torch.utils.data.DataLoader(imagenette_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

In [ ]:
imagenette.classes

In [ ]:
def get_text_embeddings(model, tokenizer, classes):
  txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt").to(device)
  text_features = model.get_text_features(**txt_inputs)
  text_emb = text_features.pooler_output.detach().to(device)
  return F.normalize(text_emb, 2, dim=1)

def get_image_embeddings(model, data_loader):
  truth = []
  image_embeddings = torch.tensor([]).to(device)

  for batch, t in tqdm(data_loader):
    with torch.inference_mode():
      img_inputs = processor(images=batch, padding=True, return_tensors='pt').to(device)
      image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to(device)
      image_emb = F.normalize(image_emb, 2, dim=1)

      image_embeddings = torch.cat((image_embeddings, image_emb), dim=0)
      truth += t

  return (image_embeddings, truth)

def display_metrics(truth, predictions, draw_plot=True):
    print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
    print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')
    print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')
    print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')
    print()

    if draw_plot:
      freqs = Counter(predictions)
      true_freqs = Counter(truth)

      fig, ax = plt.subplots(1, 2)
      fig.set_figwidth(15)
      fig.suptitle('Imagenette class sizes')

      ax[0].bar(freqs.keys(), freqs.values())
      ax[0].set_xlabel('Predicted distribution')
      ax[0].set_xticks(range(10))

      ax[1].bar(true_freqs.keys(), true_freqs.values())
      ax[1].set_xlabel('True distribution')
      ax[1].set_xticks(range(10))

# Run model on all data achievable via test loader
# Returns tuple of (predictions, truth)
def evaluate_finetuned(model, linear_layer, test_loader):
    classes_for_clip = ['A picture of ' + i[0] for i in imagenette.classes]
    predictions = []
    truth = []

    for batch, t in tqdm(test_loader):
      with torch.inference_mode():
        img_inputs = processor(images=batch, padding=True, return_tensors='pt').to(device)
        image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to(device)
        image_emb /= torch.norm(image_emb, 2, dim=1, keepdim=True)

        logits = linear_layer(image_emb) # Instead of logits = image_emb @ text_emb.T
        predicted_cat = logits.argmax(dim=1).to('cpu')

        batch_truth = torch.Tensor(t).to('cpu')

        predictions += predicted_cat.tolist()
        truth += t

    return (predictions, truth)

## Bare CLIP perfomance

In [ ]:
classes_for_clip = ['A picture of ' + i[0] for i in imagenette.classes]

predictions = []
truth = []

text_emb = get_text_embeddings(model, tokenizer, classes_for_clip)
image_emb, truth = get_image_embeddings(model, test_loader)

logits = image_emb @ text_emb.T
predictions = logits.argmax(dim=1).to('cpu').tolist()

In [ ]:
display_metrics(truth, predictions)

## Adding a layer after CLIP image processor

In [ ]:
linear_layer = torch.nn.Linear(in_features=512, out_features=10, device=device)

### Training

In [ ]:
def fit(model, processor, epochs=2):
  history = []

  optimizer = torch.optim.Adam(linear_layer.parameters())
  loss = torch.nn.CrossEntropyLoss()

  for epoch in range(epochs):
    epoch_loss = 0
    for batch, t in tqdm(train_loader):
      img_inputs = processor(images=batch, padding=True, return_tensors='pt').to(device)
      image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to(device)
      optimizer.zero_grad()
      logits = linear_layer.forward(image_emb)
      loss_val = loss(logits, torch.tensor(t).to(device))
      loss_val.backward()
      optimizer.step()
      epoch_loss += loss_val.item()

    p, t = evaluate_finetuned(model, linear_layer, test_loader)
    history.append(epoch_loss / train_loader.batch_size)
    print(f'Epoch {epoch + 1} (loss {history[-1]})')
    print()
    display_metrics(t, p, draw_plot=False)

  return history

In [ ]:
losses = fit(model, processor, epochs=4)

In [ ]:
plt.title("Loss over epochs")
plt.plot(losses)

### Evaluating

In [ ]:
predictions, truth = evaluate_finetuned(model, linear_layer, test_loader)
display_metrics(truth, predictions)